# C12-classical-models — Practice p06 — Solution


The sign-aware sigmoid never exponentiates a large positive magnitude. Stable logit BCE evaluates the same likelihood objective directly and averages exactly once.


In [ ]:
import numpy as np


def _finite_float_array_p06(value, name):
    try:
        array = np.asarray(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be numeric") from exc
    if not np.issubdtype(array.dtype, np.number) or np.iscomplexobj(array):
        raise ValueError(f"{name} must be real numeric")
    array = array.astype(np.float64)
    if array.size == 0 or not np.isfinite(array).all():
        raise ValueError(f"{name} must be finite and nonempty")
    return array


def stable_sigmoid(z: np.ndarray) -> np.ndarray:
    values = _finite_float_array_p06(z, "z")
    result = np.empty_like(values, dtype=np.float64)
    nonnegative = values >= 0.0
    result[nonnegative] = 1.0 / (1.0 + np.exp(-values[nonnegative]))
    exp_values = np.exp(values[~nonnegative])
    result[~nonnegative] = exp_values / (1.0 + exp_values)
    return result


def mean_bce_from_logits(z: np.ndarray, y: np.ndarray) -> float:
    values = _finite_float_array_p06(z, "z")
    labels = _finite_float_array_p06(y, "y")
    if values.shape != labels.shape:
        raise ValueError("z and y must have identical shape")
    if not np.all((labels == 0.0) | (labels == 1.0)):
        raise ValueError("y must contain only 0 and 1")
    losses = np.maximum(values, 0.0) - labels * values + np.log1p(np.exp(-np.abs(values)))
    return float(losses.mean())


probe_z_p06 = np.array([-10000.0, -3.0, 0.0, 3.0, 10000.0], dtype=np.float64)
probe_y_p06 = np.array([0.0, 1.0, 0.0, 1.0, 1.0], dtype=np.float64)
probabilities_p06 = stable_sigmoid(probe_z_p06)
loss_p06 = mean_bce_from_logits(probe_z_p06, probe_y_p06)


### Answer check


In [ ]:
ATOL = 1e-12
RTOL = 1e-10
expected_probability_p06 = np.array([0.0, 0.04742587317756678, 0.5, 0.9525741268224334, 1.0])
expected_loss_p06 = (np.log1p(np.exp(3.0)) + np.log(2.0) + np.log1p(np.exp(-3.0))) / 5.0
assert probabilities_p06.dtype == np.float64 and probabilities_p06.shape == probe_z_p06.shape
assert np.allclose(probabilities_p06, expected_probability_p06, atol=ATOL, rtol=RTOL)
assert np.isclose(loss_p06, expected_loss_p06, atol=ATOL, rtol=RTOL)
assert np.isclose(stable_sigmoid(np.array(0.0)).item(), 0.5, atol=ATOL, rtol=RTOL)
for bad in ([], [np.inf], [1.0 + 1.0j], ["x"]):
    try:
        stable_sigmoid(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid sigmoid input accepted")
